# Exctraction des données 
Le but de cette première partie est d'extraire les données contenues dans les pdf publiés par l'université de Lille. 
Nous cherchons notamment à retranscrire les informations que ces derniers contiennent sous forme de données tabulaires afin d'avoir une base utilisable pour la constuction de notre modèle. 
Les pdf ont toujours la même structure ce qui facilite grandement le travail d'extraction avec des outils comme pdfplumber. 

In [1]:
import re
import pandas as pd
import pdfplumber

In [2]:
COLUMNS = [
    "diplome", "intitule_emploi", "missions", "categorie_emploi", "type_emploi",
    "secteur_employeur", "activite_employeur", "lieu_emploi", "revenu",
    "autre_diplome",
]
# Définis à l'avance les infos pour chaque individu sondé qu'on va chercher à stocker dans le df

PARAS_TABLEAUX = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines"
}
# paramètre utile pour pdfplumber, c'est grâce à ça qu'il va réussir à reconnaître les cellules dans les tableaux du pdf. 
# on lui indique juste qu'ici les séparations à prendre en compte sont simplement les lignes des tableaux 


TEXT_KW = {
    "x_tolerance": 1.5,
    "y_tolerance": 2
}
# Les espaces entre les mots sont très serrés (environ 2 points).
# Une faible tolérance évite de coller plusieurs mots ensemble 

# Zone de l'en-tête. La partie droite est coupée pour exclure le bloc
# « Observatoire de la Direction des Formations ».
HEADER_BOX = (130, 0, 715, 100)

# Bande verticale contenant l'encadré de statistiques.
STATS_Y = (100, 212)
# la largeur sera celle de la page entière, je préfère ne rien fixer au cas ou les tableaux seraient parfois plus ou moins large, la hauteur peut en revanche être fixée. 

La stratégie de découpage (exemple éviter les zones inintéréssantes du pdf ou def la partie avec les stats intéressantes) vient de nous. 
En revanche j'ai fais analyser quelques extraits du pdf à un LLM pour qu'il trouve les meilleurs 'coordonées' pour faire ça. 
Par exemple c'est lui à determiné que la bande verticale avec l'encadre des stats était de taille (100,212). 
NB : J'aurai certainement, en fouillant "a la main", pu trouver ces paramètres mais je reconnais être assez mauvais pour la gestion des tailles/dimensions sur des images en python (que je fais habituellement avec cv2 en plus...)

In [3]:
HEADER_RE = re.compile(
    r"Mention\s*:\s*(?P<mention>.+?)\s*"
    r"Parcours\s*:\s*(?P<parcours>.+?)\s*"
    r"Promotion\s+(?P<promotion>\d{4})\s*-\s*Situation en\s+(?P<situation>[^( \n]+)",
    re.S,
)
# on recherche les expressions régulières (les pdf ont tjr la même structure de présentation) et on les stock dans les variables associées. 

# Un motif de recherche est associé à chaque statistique.
STATS_RE = {
    "n_interroges": r"interrog\S*\s*:\s*(\d+)",
    "taux_reponse": r"Taux de réponse\s*:\s*(\d+)\s*%",
    "n_en_emploi": r"En emploi\s+(\d+)",
    "n_recherche_emploi": r"En recherche d['’]emploi\s+(\d+)",
    "n_en_etudes": r"En études\s+(\d+)",
    "n_autre_situation": r"Autre situation\s+(\d+)",
    "n_total_repondants": r"Total\s+(\d+)",
    "n_emploi_stable": r"Emploi stable\s*:\s*(\d+)",
    "n_emploi_non_stable": r"Emploi non stable\s*:\s*(\d+)",
}

# Pour ces catégories, une valeur absente dans l'encadré signifie zéro (on va pas mettre des Nan ici ca reste une info intéressante de savoir s'il y en a 0 ou nn)
BOX_COUNTS = [
    "n_en_emploi",
    "n_recherche_emploi",
    "n_en_etudes",
    "n_autre_situation"
]


In [4]:
def clean(text):
    """Nettoie le texte d'une cellule et remplace certaines valeurs par None.
    Le but est de garantir que les données récupérées sont propres et homogènes avant d'être stockées."""
    if text is None:
        return None

    # Remplacer les suites d'espaces et de retours à la ligne par un espace.
    text = re.sub(r"\s+", " ", text).strip()

    # + considere qlq valeurs comme des données manquantes :
    if text in ("", "-", "NR"):
        return None

    return text


la ligne "text = re.sub(r"\s+", " ", text).strip()" est une suggestion de l'ia. Le LLM recommande d'appliquer ce type de normalisation car sur de l'extraction de texte à partir de pdf il est fréquent d'avoir des espèces de "sauts à la ligne parasites".

In [5]:
def parse_header(page):
    """Renvoie les informations de l'en-tête et indique si la page est une suite."""
    top = page.crop(HEADER_BOX).extract_text(**TEXT_KW) or "" # recup le texte du haut de pager
    match = HEADER_RE.search(top) # cherche les expressions regulières def plus tot

    if not match:
        return None, False # si on trv rien on ne considère pas que la page est une 'suite' mais bien une page sensée etre la première page sur cette forma ou une page sans info pertinentes

    header = {
        key: re.sub(r"\s+", " ", value).strip()
        for key, value in match.groupdict().items()
    } # recup un dictionnaire avec les infos qui nous interessent

    # La mention « (suite) » se trouve à gauche de la zone d'en-tête.
    full_top = page.crop((0, 0, page.width, 100)).extract_text(**TEXT_KW) or ""

    return header, "(suite)" in full_top
    # Grâce a cette fonction on recup les infos de l'entete et on sait normalement si c'est une page type 'suite' ou nn


In [6]:
def parse_stats(page): # Même principe que la fonction précédente mais cette fois pour la récup des stats 
    """Lit les statistiques présentes dans l'encadré de la page."""
    text = page.crop(
        (0, STATS_Y[0], page.width, STATS_Y[1])
    ).extract_text(**TEXT_KW) or "" # Recup l'encadré grace aux paras qui avaient étaient defs par le LLM

    stats = {}

    # utilisation des expressions regulières pr chercher les stats et les stocker ds le dico : 
    for key, pattern in STATS_RE.items():
        match = re.search(pattern, text)

        if match:
            stats[key] = int(match.group(1))
        else:
            stats[key] = None 

    # Une ligne absente signifie zéro pour ces catégories,
    # si le total des répondants est présent.
    if stats["n_total_repondants"] is not None:
        for key in BOX_COUNTS:
            if stats[key] is None:
                stats[key] = 0

    # idnique si le PDF signale un effectif insuffisant : très important psk certaines pages sont vides (pas assez de répondants) et c'est cette mention qui les caractérise
    full_text = page.extract_text(**TEXT_KW) or ""
    stats["effectif_insuffisant"] = "Effectif insuffisant" in full_text

    return stats


In [7]:
def extract_jobs(page):
    """Extrait les lignes du tableau des emplois d'une page."""
    rows = []

    # rechercher les tableaux présents sur la page avec les paras spécifiés a pdfplumber
    for table in page.find_tables(PARAS_TABLEAUX):

        # Ignorer les tableaux qui n'ont pas le nombre attendu de colonnes
        # ca permet notamment d'ignorer le petit tableau de statistiques
        if len(table.rows[0].cells) != len(COLUMNS):
            continue

        for row in table.rows:
            values = []

            # Lire et nettoyer chaque cellule de la ligne.
            for cell in row.cells:
                if cell:
                    value = clean(
                        page.crop(cell).extract_text(**TEXT_KW)
                    )
                else:
                    value = None

                values.append(value)
        # On utilise la force de pdfplumber ici, capable de reconnaitre et différencier les cellules de tableaux présents sur des pdf
        # et on réutilise la fonction de nettoyage def plus tot pour stocker un texte propre

            # Ne pas conserver la ligne qui contient les titres des colonnes :
            if values[0] == "Diplômé":
                continue

            # enfin on lie les vals aux bonnes colonnes :
            rows.append(dict(zip(COLUMNS, values))) # on peut utiliser zip car j'ai fais attenton de bien def les colonnes ds le bonne ordre au début

    return rows


In [8]:
def extract(pdf_path, first=1, last=None):
    """pipeline complet"""
    jobs = []
    formations = []

    current_key = None
    current_stats = {}

    with pdfplumber.open(pdf_path) as pdf:

        # si la dernière page n'est pas précisée, lire jusqu'à la fin du PDF. on ajt ca pour pouvoir tester sur qlq pages a chaque fois pr avoir une verif manuel
        if last is None:
            last = len(pdf.pages)

        for page_no in range(first, last + 1):
            page = pdf.pages[page_no - 1]

            header, is_suite = parse_header(page)

            # Ignorer les pages sans en-tête de formation (page sans infos utiles pour nous)
            if header is None:
                continue

            key = (header["mention"], header["parcours"])

            # Une page « suite » reprend les statistiques de la formation en cours.
            if is_suite and key == current_key:
                stats = current_stats

            else:
                # Nouvelle formation : lire ses statistiques.
                stats = parse_stats(page)
                current_key = key
                current_stats = stats

                formations.append({
                    "page": page_no,
                    **header,
                    **stats,
                    "n_lignes_extraites": 0
                })

            
            page_jobs = extract_jobs(page)
            formations[-1]["n_lignes_extraites"] += len(page_jobs)


            for job in page_jobs:
                jobs.append({
                    "page": page_no,
                    **header,
                    **stats,
                    **job
                })

    # transforme nos listes en 2 df : df qui est notre base globale et df_formations avec les infos spécifiques aux formations 
    df_formations = pd.DataFrame(formations)
    df = pd.DataFrame(jobs)

    
    if not df.empty:
        raw = df["revenu"].fillna("")

        # Indiquer si la valeur du revenu mentionne un temps partiel.
        df["temps_partiel"] = raw.str.contains(
            "Temps partiel",
            case=False
        )

        # Convertir le revenu en nombre. Les valeurs non numériques deviennent NaN.
        df["revenu_net"] = pd.to_numeric(
            raw.str.replace(r"\s+", "", regex=True),
            errors="coerce"
        )

        # Transformer le type d'emploi en variable booléenne.
        df["emploi_stable"] = df["type_emploi"].map({
            "Stable": True,
            "Non stable": False
        })

        # Supprimer la colonne brute, comme dans le script initial.
        df = df.drop(columns="revenu")

    return df, df_formations


In [9]:
import os
from pathlib import Path

FIRST_PAGE = 1      # première page
LAST_PAGE = None    # None = jusqu'à la fin

ROOT = Path.cwd().parents[1]                       # notebooks/extraction -> racine du dépôt
sortie_extract = ROOT / "data" / "interim" / "extract"
sortie_formations = ROOT / "data" / "interim" / "formations"

for annee in ["2023", "2024"]:
    dossier = ROOT / "data" / "raw" / annee
    for fichier in os.scandir(dossier):
        if fichier.is_file() and fichier.name.lower().endswith(".pdf"):
            nom = os.path.splitext(fichier.name)[0]
            df, df_formations = extract(fichier.path, FIRST_PAGE, LAST_PAGE)
            df.to_excel(sortie_extract / f"{nom}_extract.xlsx", index=False)
            df_formations.to_excel(sortie_formations / f"{nom}_formations.xlsx", index=False)
            print(nom, df.shape, df_formations.shape)
print("fin du traitement")

Master2023_ALL-SHS_P (817, 27) (84, 16)
Master2023_DEG_P (1241, 27) (104, 16)
Master2023_Sante-STAPS_P (408, 27) (31, 16)
Master2023_ST_P (502, 27) (66, 16)
Master2024_ALL-SHS_P (769, 27) (89, 16)
Master2024_DEG_P (1112, 27) (105, 16)
Master2024_Sante-STAPS_P (439, 27) (34, 16)
Master2024_ST_P (414, 27) (63, 16)
fin du traitement
